# Notebook 07: Non-Personalized Baseline — Global Popularity Heuristic

## 1. Purpose
Every recommender architecture must first beat a trivial popularity baseline (`ml/src/models/popularity.py`) under the identical Leave-Last-Out protocol (1 positive + 99 unobserved negatives).

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Resolve repository root deterministically from ml/notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_DIR = os.path.join(REPO_ROOT, "ml", "data", "processed")
ARTIFACTS_DIR = os.path.join(REPO_ROOT, "ml", "artifacts")
VIZ_DIR = os.path.join(REPO_ROOT, "ml", "visualizations")
plt.rcParams["figure.dpi"] = 110
from ml.src.data.dataset import RecSysDataPipeline
from ml.src.models.popularity import PopularityRecommender
from ml.src.evaluation.metrics import hit_rate_at_k, ndcg_at_k, mrr

pipeline = RecSysDataPipeline(
    articles_path=os.path.join(DATA_DIR, "articles.json"),
    interactions_path=os.path.join(DATA_DIR, "interactions.csv"),
    embeddings_path=os.path.join(DATA_DIR, "article_embeddings.npy"),
)

pop = PopularityRecommender()
pop.fit(pipeline.train_df, pipeline.num_items)

hr_l, ndcg_l, mrr_l = [], [], []
for _, row in pipeline.test_df.iterrows():
    u_idx = pipeline.user2idx[row["user_id"]]
    pos_idx = pipeline.item2idx[row["article_id"]]
    user_pos = pipeline.user_positive_items[u_idx]
    cands = [pos_idx]
    while len(cands) < 100:
        c = random.randint(0, pipeline.num_items - 1)
        if c not in user_pos and c not in cands:
            cands.append(c)
    scores = np.array([pop.item_scores.get(pipeline.idx2item[c], 0.0) for c in cands])
    ranked = [cands[i] for i in np.argsort(-scores)]
    hr_l.append(hit_rate_at_k(pos_idx, ranked, k=10))
    ndcg_l.append(ndcg_at_k(pos_idx, ranked, k=10))
    mrr_l.append(mrr(pos_idx, ranked))

print(f"Unique Popular Items Scored : {len(pop.item_scores):,}")
print(f"Popularity Baseline | HR@10 = {np.mean(hr_l):.4f} | NDCG@10 = {np.mean(ndcg_l):.4f} | MRR = {np.mean(mrr_l):.4f}")

Unique Popular Items Scored : 2,388
Popularity Baseline | HR@10 = 0.2000 | NDCG@10 = 0.0926 | MRR = 0.0861
